## 1.4. Merge DEM tiles to one GeoTIFF (optional)

This function is to merge multiple DEM files (*.tif) into one file (.tif).

# Parameters

In [1]:
import os
import glob
import rasterio
from rasterio.merge import merge

### --- Configuration (User MUST update these paths) --- ###
DEM_SOURCE_DIR = "./DEM" # all tif here should be DEM
DEM_OUTPUT_PATH = "./DEM/output/DEM_merged.tif"
### ---------------------------------------------------- ###

# Helper Functions

In [2]:
def merge_all_tiles(dem_source_dir, dem_out_path):
    os.makedirs(os.path.dirname(dem_out_path), exist_ok=True)
    tif_files = []
    for root, dirs, files in os.walk(dem_source_dir):
        for file in files:
            if file.lower().endswith('.tif'):
                tif_files.append(os.path.join(root, file))
    print(f"Found {len(tif_files)} TIFF files.")

    # Open all tiff files and add them to a list
    src_files_to_merge = []
    for fp in tif_files:
        src = rasterio.open(fp)
        src_files_to_merge.append(src)

    # Merge the rasters
    mosaic, out_trans = merge(src_files_to_merge)

    # Update metadata from one of the source files (first one in this case)
    out_meta = src_files_to_merge[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans
    })

    # Write the mosaic to disk
    with rasterio.open(dem_out_path, "w", **out_meta) as dest:
        dest.write(mosaic)
        print(f"crs is {dest.crs}")
    print(f"Merged raster saved as {dem_out_path}")


    # Close all the open datasets
    for src in src_files_to_merge:
        src.close()

# Execute Functions


In [3]:
if __name__ == '__main__':
    merge_all_tiles(DEM_SOURCE_DIR, DEM_OUTPUT_PATH)

Found 4 TIFF files.
crs is COMPD_CS["NAD83(2011) / Florida East (ftUS) + NAVD88 height (ftUS)",PROJCS["NAD83(2011) / Florida East (ftUS)",GEOGCS["NAD83(2011)",DATUM["NAD83_National_Spatial_Reference_System_2011",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1116"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","6318"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",24.3333333333333],PARAMETER["central_meridian",-81],PARAMETER["scale_factor",0.999941177],PARAMETER["false_easting",656166.667],PARAMETER["false_northing",0],UNIT["US survey foot",0.304800609601219,AUTHORITY["EPSG","9003"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","6438"]],VERT_CS["NAVD88 height (ftUS)",VERT_DATUM["North American Vertical Datum 1988",2005,AUTHORITY["EPSG","5103"]],UNIT["US survey foot",0.304800609601219,AUTHORITY["EPSG","9003"]],AXIS["Gravity-related heigh